# Feature engineering for 311 service request routing

This notebook transforms raw Calgary 311 service requests into a feature
matrix for multi-class classification. The target is `agency_responsible`
(which department handles the request). Steps include:

1. Loading and cleaning request data
2. Encoding channel and ward
3. Extracting temporal features (hour, day of week, month, year)
4. Computing community-level aggregate counts
5. Service type frequency encoding
6. Label encoding the target
7. Inspecting class distribution

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, '..')
from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features, COLUMNS_TO_KEEP
from src.model import CATEGORICAL_FEATURES, NUMERICAL_FEATURES, TARGET, prepare_model_data

DATA_DIR = str(Path('..') / 'data')
pd.set_option('display.max_columns', 40)
print('Setup complete.')

## 1. Load 311 request data

In [ ]:
raw = load_or_fetch_data(DATA_DIR, limit=100000)
print(f'Raw records: {len(raw):,}')
print(f'Columns: {list(raw.columns)}')
raw.head(3)

In [ ]:
df = preprocess_data(raw)
print(f'After preprocessing: {len(df):,} records')
print(f'Departments (target classes): {df["agency_responsible"].nunique()}')

## 2. Encode channel and ward

The `channel` field (phone, web, app, etc.) and `ward` field are categorical.
We examine their distributions before encoding.

In [ ]:
if 'channel' in df.columns:
    channel_counts = df['channel'].value_counts().reset_index()
    channel_counts.columns = ['Channel', 'Count']
    fig = px.bar(channel_counts, x='Count', y='Channel', orientation='h',
                 title='Request counts by channel',
                 color_discrete_sequence=['steelblue'])
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=350)
    fig.show()

In [ ]:
if 'ward' in df.columns:
    ward_counts = df['ward'].value_counts().head(20).reset_index()
    ward_counts.columns = ['Ward', 'Count']
    fig = px.bar(ward_counts, x='Count', y='Ward', orientation='h',
                 title='Request counts by ward (top 20)',
                 color_discrete_sequence=['darkorange'])
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=450)
    fig.show()

## 3. Extract temporal features

The `preprocess_data` function already extracts `hour`, `day_of_week`,
`month`, and `year` from `created_date`. Let us inspect their distributions.

In [ ]:
temporal_cols = ['hour', 'day_of_week', 'month', 'year']
available = [c for c in temporal_cols if c in df.columns]

fig = make_subplots(rows=1, cols=len(available), subplot_titles=available)
for i, col in enumerate(available):
    fig.add_trace(
        go.Histogram(x=df[col], name=col, marker_color='teal', showlegend=False),
        row=1, col=i+1,
    )
fig.update_layout(title='Temporal feature distributions', height=350)
fig.show()

In [ ]:
# Requests by hour and day of week heatmap
if 'hour' in df.columns and 'day_of_week' in df.columns:
    heatmap_data = df.groupby(['day_of_week', 'hour']).size().unstack(fill_value=0)
    fig = px.imshow(heatmap_data, title='Request volume by hour and day of week',
                    labels={'x': 'Hour', 'y': 'Day of week (0=Mon)'},
                    color_continuous_scale='YlOrRd')
    fig.update_layout(height=350)
    fig.show()

## 4. Community-level counts

The `engineer_features` function computes `community_request_count` and
`community_avg_resolution` as aggregate features.

In [ ]:
df = engineer_features(df)

if 'community_request_count' in df.columns:
    print(f'community_request_count range: {df["community_request_count"].min()} -- {df["community_request_count"].max()}')
    fig = px.histogram(df.drop_duplicates(subset='community'), x='community_request_count',
                       nbins=40, title='Distribution of community request counts',
                       labels={'community_request_count': 'Requests per community'},
                       color_discrete_sequence=['mediumseagreen'])
    fig.update_layout(height=350)
    fig.show()

## 5. Service type frequency

Mapping each service request type to its overall frequency acts as a
prior for how common this type of issue is city-wide.

In [ ]:
if 'service_type_frequency' in df.columns:
    print(f'service_type_frequency range: {df["service_type_frequency"].min()} -- {df["service_type_frequency"].max()}')

if 'service_request_type' in df.columns:
    type_counts = df['service_request_type'].value_counts().head(15).reset_index()
    type_counts.columns = ['Service type', 'Count']
    fig = px.bar(type_counts, x='Count', y='Service type', orientation='h',
                 title='Top 15 service request types',
                 color_discrete_sequence=['steelblue'])
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=450)
    fig.show()

## 6. Label encode target and prepare model data

The `prepare_model_data` function from `model.py` handles label encoding
of all categoricals, including the multi-class target `agency_responsible`.

In [ ]:
X, y, label_encoders, feature_names = prepare_model_data(df)

print(f'Feature matrix: {X.shape}')
print(f'Features: {feature_names}')
print(f'Target classes: {len(label_encoders["_target"].classes_)}')
print(f'\nTarget class names:')
for i, name in enumerate(label_encoders['_target'].classes_):
    print(f'  {i}: {name}')

## 7. Class distribution

Multi-class imbalance affects model performance. We inspect the target
distribution to determine whether stratified splitting is needed.

In [ ]:
target_dist = y.value_counts().sort_values(ascending=False).reset_index()
target_dist.columns = ['Encoded label', 'Count']
target_dist['Department'] = label_encoders['_target'].inverse_transform(target_dist['Encoded label'])
target_dist['Pct'] = (target_dist['Count'] / target_dist['Count'].sum() * 100).round(1)

fig = px.bar(target_dist, x='Count', y='Department', orientation='h',
             text='Pct', title='Target class distribution (agency_responsible)',
             color_discrete_sequence=['darkorange'])
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)
fig.show()

target_dist[['Department', 'Count', 'Pct']]

## Summary

Feature engineering steps completed:

1. Loaded and cleaned 311 requests, filtering to the top 15 departments.
2. Encoded `channel` and `ward` via `LabelEncoder` inside `prepare_model_data`.
3. Extracted temporal features: `hour`, `day_of_week`, `month`, `year`.
4. Computed community-level request counts and average resolution time.
5. Added `service_type_frequency` as a proxy for issue prevalence.
6. Label-encoded the multi-class target `agency_responsible`.
7. Confirmed significant class imbalance, motivating stratified splits.

The feature matrix is ready for model training in `03_modeling.ipynb`.